# 260817 cindysha 
### 1) `.gitignore`
**文件作用**：定义 Git 忽略规则。  
**本次改动**：
- 新增忽略 `.DS_Store`
- 新增忽略 `CODEBUDDY.md`
- `AGENT.md` 行尾调整（本质规则不变）

**改后功能**：
- 避免 macOS 系统文件、开发辅助文档被误提交，仓库更干净。

---

### 2) `grc/agent-backend/__init__.py`（删除）
**文件作用（原）**：旧的 `agent-backend` 包入口。  
**本次改动**：整文件删除。  
**改后功能**：
- 表示包结构迁移：从 `grc/agent-backend` 迁到 `grc/agent`（见下方新增/重命名文件）。

---

### 3) `grc/agent/__init__.py`（新增，核心）
**文件作用**：新的 Agent 高层入口。  
**本次改动**：
- 新增 `build_flow_graph_from_text(...)` 主入口：
  - 接收自然语言需求
  - 生成 FlowGraph
  - `rewrite + validate`
  - 保存为 `.grc` 并返回路径
- 新增两条生成路径：
  1. **LLM 路径**：配置了 `GRC_AGENT_*` 时，调用 `grc.agent.llm` 直接生成 `.grc YAML` 后导入
  2. **回退路径**：未配置 LLM 时生成内置 demo 流图
- 新增 YAML 解析与导入 `_flow_graph_from_grc_text`
- 新增 demo 组图 `_demo_flow_graph`

**改后功能**：
- GRC 具备“文本意图 → 自动建图 → 输出/打开 .grc”的统一后端能力。
- 即使没配 LLM，也能跑通端到端主链路（保证可用性）。

---

### 4) `grc/agent-backend/env.py` → `grc/agent/env.py`（重命名，内容未变）
**文件作用**：Agent 环境桥接/平台构建辅助。  
**本次改动**：纯路径迁移。  
**改后功能**：
- 与新包命名 `grc.agent` 对齐，便于统一导入。

---

### 5) `grc/agent-backend/examples/__init__.py` → `grc/agent/examples/__init__.py`（重命名）
**作用**：examples 子包声明。  
**改后功能**：路径归一到新包。

---

### 6) `grc/agent-backend/examples/bpsk_2g4_regression.py` → `grc/agent/examples/bpsk_2g4_regression.py`（重命名）
**作用**：示例/回归脚本。  
**改后功能**：迁移到新目录，逻辑不变。

---

### 7) `grc/agent/llm.py`（新增，核心）
**文件作用**：LLM 接入层（OpenAI-compatible Chat Completions）。  
**本次改动**：
- 新增配置读取：`GRC_AGENT_BASE_URL / API_KEY / MODEL / TIMEOUT / MAX_MESSAGES`
- 新增配置检查 `is_configured()`
- 新增系统提示词 `SYSTEM_PROMPT`（约束模型输出合法 `.grc YAML`）
- 新增消息组装（支持 history 截断）
- 新增 HTTP 调用（`urllib`，无第三方依赖）
- 新增健壮错误处理（HTTP/JSON/字段缺失/空响应）
- 新增输出后处理：去掉 ```yaml 围栏，提取纯 YAML

**改后功能**：
- Agent 可以直接通过外部大模型“按约束生成 GRC 流图文本”；
- 且具备配置化、可控超时、历史多轮上下文与容错。

---

### 8) `grc/core/blocks/options.py`
**文件作用**：GRC `options` 块与 workflow 选择逻辑。  
**本次改动**（`update_current_workflow`）：
- 从 `get_value()` 改为读参数原始值 `.value` 来匹配 workflow
- 更新断言报错信息中的取值来源

**改后功能**：
- 修复初始化早期阶段 workflow 匹配失真问题（尤其 `output_language` / `generate_options` 在首轮 rewrite 时被错误清洗导致匹配失败）。
- 直接收益：不会错误把有效语言（如 python）重置为空，workflow 更稳定。

---

### 9) `grc/core/default_flow_graph.grc`
**文件作用**：默认新建流图模板。  
**本次改动**：
- 在 options 增加：
  - `output_language: python`
  - `generate_options: qt_gui`

**改后功能**：
- 默认流图在工作流参数上更完整明确；
- 与新的 workflow 机制、GUI/生成链路更一致，减少空值导致的不确定行为。

---

### 10) `grc/gui/AgentPanel.py`（新增，核心）
**文件作用**：GTK 右侧 Agent 聊天面板。  
**本次改动**：
- 新增聊天 UI（历史区 + 输入框 + 发送按钮）
- 新增 `open_flow_graph` 信号：建图成功后通知主窗口打开 `.grc`
- 新增后台线程执行建图，避免 GTK 主线程卡死
- 支持简单多轮历史（`self._history`）

**改后功能**：
- 用户可在 GUI 里直接“用自然语言提需求”，系统自动生成并载入流图；
- 具备基本交互体验与非阻塞执行。

---

### 11) `grc/gui/Application.py`
**文件作用**：GTK 应用生命周期入口。  
**本次改动**：
- `Gtk.Application` 增加 `application_id="org.gnuradio.grc"` 和 `NON_UNIQUE` flag
- `do_activate` 中显式 `show_all()` + `present()`

**改后功能**：
- 重点修复 macOS/Quartz 下窗口不显示或进程过早退出问题；
- 应用激活与窗口映射更稳定。

---

### 12) `grc/gui/MainWindow.py`
**文件作用**：主窗口布局与面板可见性控制。  
**本次改动**：
- 引入 `AgentPanel`
- 右侧面板改为 `Gtk.Stack + StackSwitcher`：
  - `Core`（原块树）
  - `Agent`（新聊天面板）
- 增加 `_on_agent_open_flow_graph`：接收 Agent 生成路径后开新页
- 更新 panel 可见性逻辑：从 `btwin` 单体切换为 `right_top/stack` 容器控制

**改后功能**：
- 右侧栏支持“块树/Agent”标签切换，类似 IDE 侧边栏；
- Agent 生成结果可一键回灌到编辑器页签。

---

### 13) `grc/main.py`
**文件作用**：程序启动入口（GTK/QT）。  
**本次改动**（GTK 路径）：
- 引入 `from .agent import env`
- 在 build library 前调用 `env.bridge_package_name()`
- `platform.build_library()` 改为 `platform.build_library(env.block_paths())`
- 增加 `if __name__ == '__main__': main()`

**改后功能**：
- 解决“源码树 GRC + conda GNU Radio 运行时”混搭场景下的关键桥接：
  1. `grc` 包映射为 `gnuradio.grc`，使 workflow 生成模块可导入  
  2. 把源码树 `grc/blocks` 加入 block 搜索路径，确保 `*.workflow.yml` 可加载
- 启动入口更标准，可直接脚本方式执行。

---

# 这次分支改动的“整体功能结论”
**“把 Agent 能力正式接入 GRC GTK GUI 的端到端落地”**，并修复了 workflow/启动稳定性问题：

1. **能力新增**：自然语言 → LLM 生成 `.grc` → 校验保存 → GUI 自动打开  
2. **架构整理**：`agent-backend` 重命名归并到 `grc/agent`  
3. **稳定性修复**：workflow 匹配、默认 flowgraph 参数、macOS GTK 显示生命周期  
4. **运行时兼容**：source-tree + conda 混合环境桥接



# 260818 cindysha

> 启动方式（gnuradio 虚拟环境下）：
> `PYTHONPATH=$PWD python -m grc.main --gtk` 拉起 GTK 窗口，右侧 Agent 面板可交互。

## agent 目前执行的任务是什么？
**不是二选一，而是一条完整闭环**：自然语言意图 → 建图（生成可运行 `.grc`）→ 仿真（出星座/频谱/眼图）→ 按指标调参。

- **本质是“波形设计 + 仿真验证”**，生成的 `.grc` 使用仿真信源/信道（如 AWGN），可编译成 Python 跑起来看指标。
- **不是直接驱动 SDR 硬件发射射频**：要真发射需把流图里的仿真 sink 换成 USRP/osmocom 等硬件块，agent 默认走仿真路径。
- 两种工作模式（GUI 面板可切换）：
  1. **一句话直出（baseline）**：`build_flow_graph_from_text` 直接 LLM 产 `.grc`，只建图不主动仿真；
  2. **多轮协商 Agent（今天接进 GUI）**：走 planner 五阶段 `INTENT → PROPOSE → BUILD → SIMULATE → TUNE → DONE`，可建图、可仿真、可按 EVM/Eb·N0 调参。

---

### 1) `grc/agent/tools/skill_tools.py`（新增，核心）
**文件作用**：宏工具层——把 `skills` 的编排能力（`design_link`/`debug_by_metric`）暴露给 LLM function-calling。  
**本次改动**：
- 用 `@tool(group="macro")` 把 `design_link`（一步按配方建图）、`debug_by_metric`（一步按指标诊断）注册为“宏工具”
- 通过 `ctx.extra["profile"]` 桥接用户专业度画像，使宏工具也能按档位渲染叙述（narrative）

**改后功能**：
- LLM 在 BUILD 阶段可一步 `design_link` 建图、在 TUNE 阶段一步 `debug_by_metric` 诊断，无需逐块手工搭建。

---

### 2) `grc/agent/tools/registry.py`
**文件作用**：工具注册表（`@tool` 装饰器 + JSON-Schema + 调度入口）。  
**本次改动**：
- `_TOOL_MODULES` 新增 `"skill_tools"`，使宏工具在 `load_all()` 时被自动发现并注册。

**改后功能**：
- 宏工具正式进入 `macro` 分组，可导出 function-calling schema 供模型调用。

---

### 3) `grc/agent/core/planner.py`
**文件作用**：五阶段状态机，约束每个阶段允许调用的工具分组。  
**本次改动**：
- `_ALLOWED_GROUPS` 在 `BUILD` 与 `TUNE` 阶段放开 `macro` 组

**改后功能**：
- 让 LLM 在建图/调参阶段可合法调用宏工具，同时仍保持 ReAct 不越权乱调其它阶段工具。

---

### 4) `grc/agent/core/agent.py`
**文件作用**：Agent 编排内核（多轮 `step` 主循环）。  
**本次改动**：
- 每轮把 `self.ctx.profile` 注入 `tool_ctx.extra["profile"]`，使宏工具也能按档位渲染叙述（创新 B 贯穿到 function-calling 路径）
- 增强 `_merge_artifacts`：把宏工具嵌套的 `artifacts`（`grc_path`/`constellation_png`/`spectrum_png`/`eye_png`）与 `metrics` 上浮到顶层

**改后功能**：
- GUI 与实验埋点都能直接从顶层 `artifacts` 拿到 `.grc` 路径与产物图；专业度画像对宏工具生效。

---

### 5) `grc/agent/experiments/__init__.py` + `grc/agent/experiments/ablation.py`（新增）
**文件作用**：CHI 实验脚手架——用 `Session` 埋点 + `FlowGraphStore` 复用，跑量化消融。  
**本次改动**：新增三条量化实验
- **E1 自适应 vs 固定档位**：同一段“专业度渐变”对话，对比 `adaptive` 开/关下的档位轨迹（断言“非降且末档 > 首档”）
- **E2 三档表达差异化**：同一 `design_link` 结果在 novice/student/expert 三档下叙述的差异度（0.68~0.82）
- **E3 经验复用**：`FlowGraphStore` 记住一次建图后，相似意图能否召回同一配方省往返

**改后功能**：
- 为论文提供可复现、可量化的消融证据；三条实验全部 PASS。

---

### 6) `grc/gui/AgentPanel.py`（重写，核心）
**文件作用**：GTK 右侧 Agent 聊天面板。  
**本次改动**：
- 从旧“一句话直出”升级为**多轮协商**：持有 `Agent` 实例逐轮 `agent.step`，回显按档位渲染的叙述
- **内联展示产物图**：把 `artifacts` 里的星座/频谱/眼图缩略显示在面板内
- 产出 `.grc` 时 emit `open_flow_graph` 信号让主窗口载入画布
- 新增**专业度档位下拉**（自适应/小白/学生/专家，体现创新 B：可手动钉档或让其自适应）
- 保留 **baseline 开关**（“一句话直出”）作为论文对照
- 后台线程执行、非阻塞 GTK 主循环

**改后功能**：
- 用户在 GUI 里即可完成“提需求 → 看叙述 → 看产物图 → 一键载入流图”的多轮闭环，并可切换 baseline 做对照。

---

# 这次改动的“整体功能结论”
**“把 skills 编排能力升级为宏工具并贯通到 LLM/GUI，形成可量化验证的多轮协商闭环”**：

1. **能力升级**：宏工具（`design_link`/`debug_by_metric`）接入 registry/planner/agent，LLM 可一步建图/一步诊断
2. **GUI 升级**：AgentPanel 走多轮协商 + 内联产物图 + 档位控件，保留 baseline 对照
3. **实验支撑**：新增 ablation E1/E2/E3 三条量化消融，全部 PASS
4. **产物贯通**：宏工具嵌套产物/指标上浮，GUI 与埋点统一可读



# 260818 cindysha

## 现象
小白档下输入"给我一个最简单的例子，一个正弦单音加点噪声，看看频谱"，Agent 回复"（已达工具调用步数上限）"。

> 附带的两条终端日志均无害：
> - `Gtk-WARNING: Could not load a pixbuf ... bullet-symbolic.svg` —— conda 环境 GTK/Adwaita 图标主题缺 pixbuf loader，纯渲染警告。
> - `TSM AdjustCapsLockLED... / IMKCFRunLoopWakeUpReliable` —— macOS 输入法框架（TSM/IMK）系统级日志，与本程序无关。

## 根因（编排 bug，非网络/环境问题）
多轮 Agent 首阶段是 **INTENT（意图澄清）**，本该纯自然语言复述需求、请用户确认，**不需要调工具**。但原逻辑给该阶段挂了 `knowledge` 组检索工具（`search_blocks`）并强制走 function-calling 循环。GLM 习惯性反复调检索（连调 8 次）却从不产出文本，`_run_toolcall_loop` 跑满 `max_tool_steps` 后兜底抛出"（已达工具调用步数上限）"这句生硬文案。

---

### `grc/agent/core/agent.py`（本次唯一改动文件，核心）
**文件作用**：Agent 编排内核（多轮 `step` 主循环、function-calling / 文本 ReAct 两条执行路径、system prompt 组装）。  
**本次改动**：
- **意图/方案阶段不再传工具**：`_step_with_llm` 中判断 `stage in (INTENT, PROPOSE)` 时直接调 `_force_text_reply` 一次性出自然语言回复，从机制上杜绝"只有检索工具时反复空转"（最关键的一处）。
- **新增 `_force_text_reply`**：不带 `tools` 请求一次、强制模型给自然语言回复。同时服务两个场景：(1) 纯协商阶段的一次性直出；(2) tools 循环耗尽后的兜底总结。用 `list(messages)` 局部副本追加一条 system 引导语，**不污染调用方 messages**（避免连续两条 user）。
- **循环耗尽兜底更健壮**：`_run_toolcall_loop` 与 `_run_react_text` 跑满步数时，不再返回死板的"（已达工具调用步数上限）"，改为调 `_force_text_reply` 逼模型基于已有 observation 给一句真回复。
- **`max_tool_steps` 6 → 8**：给 BUILD/SIMULATE 等真需要多步工具的阶段留余量。
- **system prompt 分阶段职责化**：`_system_prompt` 按 `stage` 注入分阶段说明（intent/propose 以自然语言协商为主、工具可选勿空转；build/simulate/tune 才真正调工具），并强调"每轮最终都必须给出一句面向用户的自然语言回复"。

**改后功能**：
- INTENT/PROPOSE 阶段 **0 次工具调用**、直接出贴合档位的自然语言回复，`needs_confirmation=True` 正常。
- 即便在 BUILD/SIMULATE 阶段跑满步数，也能兜底拿到一句真回复而非"上限"文案。
- 彻底消除"（已达工具调用步数上限）"。

---

## 验证
实跑复现场景（小白档 + 那句需求）：
- INTENT 阶段：`TOOLS: []`，直接出小白档意图复述，`needs_confirm=True`。
- 回复"对，就是这个意思" → 进入 PROPOSE：同样 `TOOLS: []`，直接给出"三步走"方案。
- 不再出现"（已达工具调用步数上限）"。

---

# 这次改动的"整体功能结论"
**"修复多轮 Agent 意图/方案阶段的工具空转 bug，让纯协商阶段稳定直出自然语言回复"**：

1. **机制修复**：INTENT/PROPOSE 不传工具、一次性直出，杜绝空转（根治）
2. **兜底加固**：新增 `_force_text_reply`，循环耗尽也能出真回复而非生硬文案
3. **提示优化**：system prompt 分阶段职责 + 强制每轮产出自然语言
4. **余量调整**：`max_tool_steps` 6 → 8，兼顾多步工具阶段



# 260819 cindysha

> 启动方式（gnuradio 虚拟环境）：
> `cd <项目根> && /path/to/envs/gnuradio/bin/python -m grc --gtk`，右侧 Agent 面板可交互。

## 本次架构的核心变化
**把 Agent 内核从旧的自研 `core.Agent + planner 五阶段` 迁移为基于 `deepagents` 的多智能体装配层 `grc/agent/service/`。** 旧 `grc/agent/core/`、`grc/agent/experiments/`、`memory/session.py`、`memory/store.py` 被移除/外迁（消融脚本移到根 `scripts/`），Agent 主链路改由 `ServiceAgent` 编排。

整体形成**三条产出路径**，共用同一套确定性工具，保证任何环境都能出图：
- **主路径**：`ServiceAgent` → `deepagents.create_deep_agent`（主 Agent + 4 个专职 subagent + SKILL 渐进式披露）
- **降级路径**：未装 `deepagents` 或未配 LLM 时，自动回落到确定性 `design_link` 宏建图（论文 baseline，无 LLM 也出图）
- **对照路径**：GUI 勾选“一句话直出(baseline)” → `build_flow_graph_from_text`，LLM 直接产 `.grc` YAML

---

## 分层结构（迁移后）
```
表现层  grc/gui/          AgentPanel(面板/档位/内联图) · MainWindow(挂载 + open_flow_graph)
契约层  grc/agent/schema.py                 AgentReply / ToolInvocation（GUI 渲染零耦合内部实现）
装配层  grc/agent/service/  ★主路径核心
        adapter.py        ServiceAgent：GUI 契约守门人 / 主·降级路由 / 结果折叠
        orchestrator.py   build_agent()：create_deep_agent 组装深度代理
        model.py          llm.get_config() → LangChain ChatOpenAI
        tools_lc.py       确定性工具 → LangChain @tool（单一事实源桥接）
        subagents.py      4 个 SubAgent（知识 / 建图 / 校验 / 仿真）
        system_prompt.py  主 Agent 阶段编排 + 各 subagent 角色 prompt + STYLE 段
        backend.py        CompositeBackend：State 产物区 + SKILL 只读挂载
        session_store.py  会话落盘镜像 + events.jsonl 事件流
工具层  grc/agent/tools/  registry(@tool 注册/调度) · design_link(宏建图·降级底座) · build/critic/sim...
料 层  knowledge/recipes.py(配方库) · memory/profile.py(三档画像) · runtime(无头仿真) · skills(SKILL md)
基座层  llm.py(GRC_AGENT_* 接入) · env.py(桥接/make_platform) · grc.core(不修改)
```

---

### 1) `grc/agent/service/adapter.py`（新增，核心 · `ServiceAgent`）
**文件作用**：GUI 与内核之间的守门人；`step(text)` 是总调度。  
**本次改动**：
- 新增 `ServiceAgent.step()`：`profile.observe` 更新档位 → `_make_ctx` 组装共享 `ToolContext` → `orchestrator.build_agent` 选路
- **主/降级路由**：深度代理可用走 `_run_deep`，否则 `_run_deterministic` 直调 `design_link`
- `_fold` 把过程事件/产物折叠为稳定的 `AgentReply`，并补扫 `final/*.grc` 得到 `grc_path`
- 统一异常收敛为 `stage=ERROR` 的 `AgentReply`，GUI 侧零改动

**改后功能**：
- GUI 只依赖 `agent.step(text) -> AgentReply` 一个契约；主/降级/报错对界面完全透明。

---

### 2) `grc/agent/service/orchestrator.py`（新增，核心）
**文件作用**：用 `deepagents.create_deep_agent` 组装深度代理。  
**本次改动**：
- `build_agent(ctx)` 装配五要素：model / tools（4 个 LangChain 工具）/ subagents / skills / backend + `InMemorySaver` checkpointer
- 缺 `deepagents` 或 LLM 不可用时返回 `None`，触发 adapter 降级

**改后功能**：
- 主路径成为真正的多智能体编排（主 Agent 按阶段用内置 `task` 委派 subagent）。

---

### 3) `grc/agent/service/subagents.py` + `system_prompt.py`（新增）
**文件作用**：4 个专职子代理 + 各自角色 prompt。  
**本次改动**：装配
- `block_knowledge_agent`（查块/解释端口，绑 SKILL `grc-block-rag`，只读）
- `flowgraph_builder_agent`（选配方建图，绑 `grc-build` + `design_flowgraph`/`validate_flowgraph`）
- `flowgraph_critic_agent`（校验并整理修复建议，绑 `grc-critic` + `validate_flowgraph`）
- `simulation_agent`（无头仿真读 EVM/BER + 画图，绑 `grc-sim` + `run_simulation`/`read_metric`）
- 主 Agent 阶段编排 `INTENT → RETRIEVE → BUILD → CRITIC → SIMULATE → DELIVER`，STYLE 段按 `profile.style_prompt()` 动态注入（创新 B）

**改后功能**：
- 职责分离的多智能体协作；同一后端、三档表达在 prompt 层落地。

---

### 4) `grc/agent/service/tools_lc.py` + `model.py` + `backend.py`（新增）
**文件作用**：把内核桥接进 deepagents 生态。  
**本次改动**：
- `tools_lc`：把确定性工具封成 4 个 LangChain `@tool`（`design_flowgraph`/`validate_flowgraph`/`run_simulation`/`read_metric`），**内部仍走 `registry`/`design_link`（单一事实源，红线 4）**
- `model`：`llm.get_config()` → `ChatOpenAI`，`is_available()` 探测
- `backend`：`CompositeBackend` = `StateBackend`（会话产物随 checkpointer 持久化）+ 把磁盘 `skills/` 只读挂载进虚拟文件系统

**改后功能**：
- 模型路径与离线降级路径产出一致，可复现；SKILL 支持渐进式披露。

---

### 5) `grc/agent/service/session_store.py`（新增）
**文件作用**：会话产物落盘 + 事件流。  
**本次改动**：
- 落盘 `local/agent_sessions/<id>/{work,final}/` 与 `events.jsonl`
- `mirror_session_files` 把 State 里的虚拟文件镜像到磁盘；`append_session_event` 记录 CHI 埋点

**改后功能**：
- 过程可追溯、可复现，实验数据源统一。

---

### 6) `grc/agent/__init__.py` / `schema.py`（调整）
**文件作用**：顶层入口与 GUI 契约。  
**本次改动**：
- 顶层惰性暴露 `ServiceAgent` / `build_service_agent`；保留 `build_flow_graph_from_text` 作为 baseline 薄包装
- `schema.py` 固化 `AgentReply` / `ToolInvocation` / `ExpertiseLevel`

**改后功能**：
- 主链路入口切到 service 层，同时保留一句话直出对照。

---

### 7) `grc/gui/AgentPanel.py`（调整）
**文件作用**：GTK 右侧 Agent 面板。  
**本次改动**：
- `_ensure_agent()` 改走 `service.build_service_agent()`（`ServiceAgent`），不再用旧 `core.Agent`
- 注入 `platform` / `out_dir=local/output/`；专业度档位 → `profile.pin/unpin` + `ctx.adaptive`
- 子线程跑 `agent.step`，`_on_agent_reply` 回显叙述 + 内联产物图，产出后 emit `open_flow_graph`

> 注：面板顶部 docstring 仍描述旧 `core.Agent + planner 五阶段`，属历史遗留注释，以代码为准。

**改后功能**：
- 面板对接新内核，主/降级对用户透明；产物统一落 `local/output/`。

---

## 用户输入 → 底层文件调用（主路径时序）
```
AgentPanel._on_send → 子线程 _handle_agent
  → service.build_service_agent()（注入 platform / out_dir / 档位）
  → ServiceAgent.step(text)   [adapter.py]
       ① memory/profile.py observe 更新档位
       ② _make_ctx（env.make_platform 如需 + session_store 目录）
       ③ orchestrator.build_agent → model/llm 探测 → create_deep_agent
       ④(主) _run_deep：主 Agent 按阶段 task 委派 subagent，subagent 调 @tool
             tools_lc.design_flowgraph → tools/design_link.py
               → knowledge/recipes.py 选配方
               → tools/registry.call: init_flow_graph/add_block/connect/validate/render_grc/run_simulation/read_metric/plot_*
               → grc.core 真实 FlowGraph 装配与存盘
         (降级) _run_deterministic：直调 design_link，同样经 recipes→registry→grc.core
       ⑤ session_store 镜像产物 + 事件；_fold → AgentReply（补扫 grc_path）
  → GLib.idle_add(_on_agent_reply)：显示叙述/产物图/指标，emit open_flow_graph
  → MainWindow._on_agent_open_flow_graph → new_page(grc_path) 载入画布
```

**产物落盘**：`local/output/`（用户可见 `.grc` 与产物图）；`local/agent_sessions/<id>/{work,final,events.jsonl}`（过程/最终/事件，属运行期产物应 gitignore）。

---

# 这次改动的“整体功能结论”
**“把 Agent 内核从自研 planner 五阶段升级为 deepagents 多智能体装配层，主/降级共用确定性工具、GUI 契约零改动”**：

1. **架构升级**：新增 `service/`（adapter/orchestrator/model/tools_lc/subagents/system_prompt/backend/session_store），移除旧 `core/`、`experiments/`
2. **多智能体**：主 Agent 阶段编排 + 4 个专职 subagent + SKILL 渐进式披露
3. **可复现红线**：主路径工具与离线降级共用 `registry`/`design_link`（单一事实源）
4. **契约稳定**：GUI 仅依赖 `step(text) → AgentReply`，主/降级/报错全透明；配套文档见 `local/docs/agent_architecture_overview.md`



# 260820 cindysha

> 启动：`conda activate gnuradio && PYTHONPATH=$PWD python -m grc --gtk`  
> 右侧 Agent 面板**不要勾选**「一句话直出」；对话区下方是 Claims / Evidence 面板。

## 本次架构的核心变化
在 260819 的 `service/` + deepagents 装配之上，补齐目标架构缺失的三块，并把 4 个专职子代理重组为**严格 6 个 Domain Subagent**：

1. **L3 Shared State**：`grc/agent/state/`，内存 dataclass + `state.json` 原子落盘
2. **Policy Gateway**：改图/换配方前 `gate()` → ALLOW / PROPOSE / DENY / CONFIRM
3. **Claim-Evidence**：成功条件登记为 Claim，仿真指标写成 Evidence，改图后按版本失效
4. **Workspace**：AgentPanel 嵌入只读 `ClaimsPanel`

主路径仍是 `ServiceAgent.step()`，不是 LLM 直接写 `.grc`：

```
用户文本
  → SharedState（规格 / 工程 / claims / 协调）
  → PolicyGateway
  → registry.call / design_link
  → AgentReply{text, artifacts, claims, spec_digest}
  → AgentPanel + ClaimsPanel + Flowgraph 画布
```

无 deepagents / 无 LLM 时，同一套 State 与工具链降级走 `design_link`。

---

## 六层结构（对照当前代码）

```
L6  Workspace     AgentPanel · ClaimsPanel · Flowgraph 画布
L5  MainAgent     ServiceAgent：确认闭环 → commit_intent → deepagents 或 design_link
                  主 Agent 持有全量工具，也可委派 6 个子代理
L4  Subagents     spec · radio_design · flowgraph · verification · diagnosis · hardware
L3  Shared State  RadioSpec / ProjectState / ClaimStore / Coordination + gate()
                  local/agent_sessions/<id>/state.json 与 snapshots/v{N}/
L2  Tools         registry.call / design_link（LLM 与降级同一执行入口）
L1  GNU Radio     env.make_platform · runtime 无头仿真
```

一轮 `ServiceAgent.step()`：损坏停写 → 画像 → 确认/取消 → `commit_intent` 抽规格与 EVM claim → 主路径 invoke（步数上限 150，撞限按已有产物交付）或降级 `design_link` → `_fold` 填 claims/spec_digest → 原子写 `state.json`。

---

### 1) `grc/agent/state/`（新增，核心）
**文件作用**：共享事实层。  
**本次改动**：
- `shared_state.py`：`RadioSpec` / `ProjectState` / `Claim`+`Evidence` / `Coordination` / `TaskCard` / `ResultEnvelope` / `SharedState`
  - 原子 `save`；损坏 JSON 备份为 `state.json.corrupt.<ts>` 并拒绝再覆盖
  - `spec_digest()` 给 GUI / Spec 导出
- `claim_store.py`：upsert、追加 Evidence（Passed/Failed/Inconclusive）、`invalidate_by_version`
- `policy.py`：`gate()` 四档
- `snapshot.py`：`snapshots/v{version}/`（原 `.grc` 文件名 + `state.json`）；`restore_snapshot` 已实现但未接 GUI

**改后功能**：
- 可追溯规格、版本化 Claim-Evidence、改前快照有物理载体。

---

### 2) `grc/agent/tools/state_tools.py` + `design_link.py`（新增/接入）
**文件作用**：把 State / Policy 接到确定性工具链。  
**本次改动**：
- `commit_intent`：从用户文本抽 modulation / channel / `EVM < n%`
- `verify_state_claims`：把 `evm_pct` 绑到 sim 层 EVM claim
- `apply_grc_diff`：单块改参；先 snapshot，再 version+1 并失效旧 claim（仅当前内存流图）
- `configure_sdr` / `list_devices`：一期只写 `config["device"]`，枚举禁用
- `design_link`：换已有配方 → `PROPOSE` 写入 `pending_confirmations`；锁定调制冲突 → `DENY`；成功建图后 version+1 + 验 claim

**改后功能**：
- 确定性路径即可演示 BPSK/EVM Claim-Evidence；配方切换必须用户确认。

---

### 3) `grc/agent/service/`（重组为 6 Agent + 确认闭环）
**文件作用**：主路径装配。  
**本次改动**：
- `subagents.py`：4 → 6  
  `spec_agent` / `radio_design_agent` / `flowgraph_agent` / `verification_agent` / `diagnosis_agent` / `hardware_agent`  
  （旧知识检索并入 RadioDesign，仿真并入 Verification）
- `system_prompt.py`：编排从固定 6 阶段改为闭环路由  
  `build | diagnose | modify | observe | spec`；prompt 要求 JSON TaskCard / ResultEnvelope（运行时尚未强校验）
- `tools_lc.py`：补 `spec_*` / `select_recipe` / `verify_claims` / `plot_spectrum` / `diagnose_by_metric` / `apply_grc_diff` / `configure_sdr`
- `adapter.py`：每轮 `resolve_confirmation` + `commit_intent`；`AgentReply` 增 `claims` / `spec_digest`；`stage` 含 `CONFIRM` / `DENY` / `CANCELLED`；撞 recursion 按已产出交付
- `session_store.py`：`state_path` / `snapshots_dir` / `export_spec` / `import_spec`（后者无 GUI 入口）
- SKILL 新增 `grc-spec` / `grc-diagnosis` / `grc-hardware`；全部全局挂到 `/workspace/skills/`

**改后功能**：
- GUI 契约仍是 `step(text) → AgentReply`；主/降级都写 SharedState；换配方走确认闭环。

---

### 4) `grc/agent/schema.py` + `grc/gui/ClaimsPanel.py` + `AgentPanel.py`
**文件作用**：GUI 契约与只读事实面板。  
**本次改动**：
- `AgentReply` 追加 `claims`、`spec_digest`（带默认值，向后兼容）
- 新增 ClaimsPanel：表 `statement | layer | status | version`；详情区默认 Spec 摘要，点选 claim 看 Evidence JSON
- AgentPanel 嵌入该面板；baseline 一句话直出仍清空 claims（不经 State）

**改后功能**：
- 对话区能同时看到叙述、产物图和 Claim-Evidence，不再只有纯文本。

---

### 5) `grc/agent/knowledge/recipes.py`
**本次改动**：新增 `rx_bpsk_awgn`（定时恢复 + 星座接收判决）。  
**已知缺口**：该配方 `metrics=[]`，不会自动出 BER/EVM 接收质量 Claim。

---

## 与 260819 笔记的差异（避免沿用过时描述）

| 260819 | 现在 |
|---|---|
| 4 个 subagent（知识 / 建图 / 校验 / 仿真） | **6 个**：Spec / RadioDesign / Flowgraph / Verification / Diagnosis / Hardware |
| 主 Agent 固定阶段 `INTENT → RETRIEVE → BUILD → CRITIC → SIMULATE → DELIVER` | prompt 级闭环路由；运行时无状态机强制 |
| 无 SharedState | `state.json` + ClaimStore + Policy + snapshot |
| AgentReply 只有叙述/产物 | 增加 `claims` / `spec_digest`，ClaimsPanel 展示 |
| 换配方直接建图 | 已有工程换配方 → `CONFIRM`，需用户「确认」或「取消修改」 |

**仍未做完**：TaskCard/ResultEnvelope 运行时校验、`active_task`、磁盘任意 `.grc` diff、锁定/回滚/Spec 导入导出的产品入口、rx 接收质量 Claim、专属 SKILL 真正写入 subagent 配置。

---

# 这次改动的“整体功能结论”
**“在 deepagents 装配层上落地 Shared State、Policy 确认与 6 个 Domain Subagent，使可追溯规格 / Claim-Evidence / 协调治理有代码载体”**：

1. **事实层**：`grc/agent/state/` + 会话 `state.json` / `snapshots/v{N}/`
2. **治理**：换配方 PROPOSE、锁定调制 DENY、改图前快照、改图后 claim 失效
3. **多智能体**：6 个 Domain Agent + 全局 SKILL；主 Agent 仍可直接 `design_flowgraph`
4. **GUI**：ClaimsPanel 只读展示；契约仍是 `AgentReply`
5. **可复现红线不变**：建图/仿真只经 `registry` / `design_link`
